<p align='center'>
<img src='https://img.shields.io/badge/MiniLLM-v2-blue?style=for-the-badge'/>
<img src='https://img.shields.io/badge/DevLab-bono--p-orange?style=for-the-badge'/>
<a href='https://colab.research.google.com/github/bono-p/minillm_v2/blob/main/MiniLLM_v2_DevLab.ipynb'>
<img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>
</p>

# 🧠 MiniLLM v2 — DevLab

**Transformer decoder-only** optimisé et entraînable sur GPU Colab gratuit (T4/A100).

| Composant | Technologie | Avantage |
|-----------|-------------|----------|
| Positional emb. | **RoPE** | Extrapolation longueur |
| Normalisation | **RMSNorm** | +10% vitesse vs LayerNorm |
| Activation FFN | **SwiGLU** | Surpasse GELU |
| Attention | **GQA** | Réduit KV cache |
| Flash Attention | **PyTorch 2.0+** | Sans lib externe |

---

## 🗺️ Plan du notebook

| # | Section | Description |
|---|---------|-------------|
| 1️⃣ | **Setup** | GPU · Drive · Dépendances · Repo |
| 2️⃣ | **Architecture** | Inspecter et choisir la taille du modèle |
| 3️⃣ | **Datasets** | Wikipedia FR (pré-entraînement) + PIAF (QA) |
| 4️⃣ | **Tokenisation** | Préparer les données binaires |
| 5️⃣ | **Pré-entraînement** | Entraîner depuis zéro |
| 6️⃣ | **Fine-tuning QA** | Spécialiser sur les données PIAF |
| 7️⃣ | **Génération** | Tester le modèle interactivement |
| 8️⃣ | **Sauvegarde** | Exporter vers Drive / télécharger |

> ⚡ **Avant de commencer** : `Exécution > Modifier le type d'exécution > GPU (T4)`
> Exécute les cellules **dans l'ordre**, de haut en bas.


---
## 1️⃣ Setup


In [ ]:
# @title 🖥️ 1.1 — Vérification GPU
import subprocess, torch
res = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free',
                      '--format=csv,noheader'], capture_output=True, text=True)
if res.returncode == 0:
    print('✅ GPU détecté :', res.stdout.strip())
else:
    print('❌ Aucun GPU ! → Exécution > Modifier le type exécution > GPU')
    raise SystemExit('GPU requis')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'BF16 ok  : {torch.cuda.is_bf16_supported()}')
DEVICE = 'cuda'


In [ ]:
# @title 📦 1.2 — Installation des dépendances
%%capture
!pip install tiktoken datasets -q
import tiktoken, datasets, numpy, torch
print(f'✅ tiktoken {tiktoken.__version__}')
print(f'✅ datasets {datasets.__version__}')
print(f'✅ numpy    {numpy.__version__}')
print(f'✅ torch    {torch.__version__}')


In [ ]:
# @title 💾 1.3 — Montage Google Drive
from google.colab import drive
import os
drive.mount('/content/drive')

# Dossier de base sur ton Drive
DRIVE_BASE   = '/content/drive/MyDrive/MiniLLM_v2'
CKPT_PRETRAIN = f'{DRIVE_BASE}/checkpoints/pretrain'
CKPT_FINETUNE = f'{DRIVE_BASE}/checkpoints/finetune'
DATA_DIR      = f'{DRIVE_BASE}/data'

for d in [CKPT_PRETRAIN, CKPT_FINETUNE, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Google Drive monté')
print(f'   Checkpoints pré-entraînement : {CKPT_PRETRAIN}')
print(f'   Checkpoints fine-tuning      : {CKPT_FINETUNE}')
print(f'   Données                      : {DATA_DIR}')


In [ ]:
# @title 📂 1.4 — Clonage du repo GitHub
import os, sys
REPO_DIR = '/content/minillm_v2'

if os.path.exists(REPO_DIR):
    print('Mise à jour du repo...')
    !cd {REPO_DIR} && git pull -q
else:
    print('Clonage du repo bono-p/minillm_v2...')
    !git clone https://github.com/bono-p/minillm_v2.git {REPO_DIR} -q

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f'✅ Repo prêt dans {REPO_DIR}')
print('Fichiers :', sorted(f for f in os.listdir('.') if f.endswith('.py') or f.endswith('.md')))


---
## 2️⃣ Architecture du modèle
> Choisis la taille de modèle adaptée à ton GPU avant de commencer.


In [ ]:
# @title 📊 2.1 — Tableau comparatif des presets
from config import PRESETS
print(f'  {"Size":>6} | {"Params":>8} | {"Layers":>6} | {"d_model":>7} | '
      f'{"Heads":>5} | {"KV":>4} | {"FFN":>5} | {"Contexte":>8}')
print('  ' + '─'*62)
for name, cfg in PRESETS.items():
    n    = cfg.count_params()
    typ  = 'MHA' if cfg.kv_heads == cfg.n_heads else f'GQA×{cfg.n_heads//cfg.kv_heads}'
    vram = n*4*4/1e9
    flag = '⚡T4' if vram < 14 else ('⚠️24G' if vram < 24 else '🔴A100')
    print(f'  {name:>6} | {n/1e6:>6.1f}M | {cfg.n_layers:>6} | {cfg.d_model:>7} | '
          f'{cfg.n_heads:>5} | {typ:>4} | {cfg.ffn_hidden:>5} | {cfg.max_seq_len:>5}  {flag}')
print()
print('  ⚡T4 = OK sur Colab gratuit | ⚠️ = GPU Pro | 🔴 = A100')


In [ ]:
# @title 🔬 2.2 — Inspecter un modèle + test forward
import torch
from config import PRESETS
from model  import MiniLLM

MODEL_SIZE = '50M'  # @param ['15M', '50M', '125M', '350M']

cfg   = PRESETS[MODEL_SIZE]
model = MiniLLM(cfg)
n     = model.n_params

n_emb  = cfg.vocab_size * cfg.d_model
n_blks = sum(p.numel() for nm,p in model.named_parameters() if 'blocks' in nm)
print(f'{'═'*52}')
print(f'  MiniLLM-{MODEL_SIZE}  ({n/1e6:.2f}M paramètres)')
print(f'{'═'*52}')
print(f'  ├── Embedding  : {n_emb/1e6:.2f}M  ({n_emb/n*100:.0f}%)')
print(f'  ├── Blocs ×{cfg.n_layers}  : {n_blks/1e6:.2f}M  ({n_blks/n*100:.0f}%)')
print(f'  └── LM Head   : {"partagé" if cfg.tie_embeddings else "séparé"}')
print(f'  Mémoire entraînement : ~{n*4*4/1e9:.1f} GB')
print()
x = torch.randint(0, cfg.vocab_size, (2, 64))
with torch.no_grad():
    logits, loss = model(x, x)
import math
print(f'  ✅ Forward pass OK')
print(f'  ✅ Loss initiale : {loss.item():.3f}  (attendu ≈ {math.log(cfg.vocab_size):.2f})')
print(f'{'═'*52}')

# Retenir la config choisie pour la suite
CHOSEN_SIZE = MODEL_SIZE
print(f'\n  → Modèle retenu : MiniLLM-{CHOSEN_SIZE}')


---
## 3️⃣ Téléchargement des datasets
> **Deux datasets** : Wikipedia FR pour le pré-entraînement, PIAF pour le fine-tuning QA.
> Les fichiers sont mis en cache sur Drive pour ne pas les re-télécharger.


In [ ]:
# @title 📰 3.1 — Wikipedia français (pré-entraînement)
import os, json
from datasets import load_dataset

WIKI_FRACTION  = '5%'   # @param ['1%','2%','5%','10%','20%','100%']
WIKI_CACHE     = f'{DATA_DIR}/wikipedia_fr_{WIKI_FRACTION.replace("%","p")}.txt'

if os.path.exists(WIKI_CACHE):
    size_mb = os.path.getsize(WIKI_CACHE)/1e6
    print(f'✅ Cache trouvé sur Drive : {WIKI_CACHE}  ({size_mb:.0f} MB)')
else:
    print(f'Téléchargement Wikipedia FR ({WIKI_FRACTION})...')
    wiki = load_dataset(
        'wikipedia', '20220301.fr',
        split=f'train[:{WIKI_FRACTION}]',
        trust_remote_code=True
    )
    print(f'  {len(wiki):,} articles téléchargés')
    print(f'  Écriture dans {WIKI_CACHE}...')
    total_chars = 0
    with open(WIKI_CACHE, 'w', encoding='utf-8') as f:
        for i, art in enumerate(wiki):
            text = art['title'] + '\n\n' + art['text'] + '\n\n'
            f.write(text)
            total_chars += len(text)
            if (i+1) % 5000 == 0:
                print(f'  {i+1:,} articles — {total_chars/1e6:.0f} MB', end='\r')
    size_mb = os.path.getsize(WIKI_CACHE)/1e6
    print(f'\n✅ Wikipedia FR sauvegardé : {size_mb:.0f} MB | {total_chars/1e6:.0f}M caractères')

WIKI_FILE = WIKI_CACHE
# Aperçu
with open(WIKI_FILE, encoding='utf-8') as f:
    preview = f.read(400)
print(f'\nAperçu :\n{"─"*40}\n{preview}\n{"─"*40}')


In [ ]:
# @title ❓ 3.2 — PIAF : dataset QA français
import os, json
from datasets import load_dataset

PIAF_CACHE = f'{DATA_DIR}/piaf_qa.json'

if os.path.exists(PIAF_CACHE):
    with open(PIAF_CACHE, encoding='utf-8') as f:
        piaf_data = json.load(f)
    print(f'✅ PIAF cache trouvé : {len(piaf_data)} exemples')
else:
    print('Téléchargement PIAF (etalab-ia/piaf)...')
    piaf = load_dataset('etalab-ia/piaf', split='train', trust_remote_code=True)
    print(f'  {len(piaf)} paires Q/R téléchargées')

    # Convertir au format texte pour notre LM
    # Format : "Contexte : ...
 Question : ...
 Réponse : ..."
    piaf_data = []
    for ex in piaf:
        answers = ex['answers']['text']
        if not answers:
            continue
        entry = {
            'context':  ex['context'][:1000],   # tronquer les contextes trop longs
            'question': ex['question'],
            'answer':   answers[0],
            'title':    ex.get('title',''),
        }
        piaf_data.append(entry)

    with open(PIAF_CACHE, 'w', encoding='utf-8') as f:
        json.dump(piaf_data, f, ensure_ascii=False, indent=1)
    print(f'✅ PIAF sauvegardé : {len(piaf_data)} exemples dans {PIAF_CACHE}')

# Aperçu de 3 exemples
print(f'\n── Exemples PIAF ──────────────────────────────────')
for ex in piaf_data[:3]:
    print(f'  Q : {ex["question"]}')
    print(f'  R : {ex["answer"]}')
    print()


---
## 4️⃣ Tokenisation
> Conversion des textes bruts en fichiers binaires `.bin` pour l'entraînement.


In [ ]:
# @title 🔡 4.1 — Tokeniser Wikipedia FR (pré-entraînement)
import sys, os
sys.path.insert(0, '/content/minillm_v2')
from prepare_data import prepare

PRETRAIN_BIN  = f'{DATA_DIR}/pretrain'
os.makedirs(PRETRAIN_BIN, exist_ok=True)

train_bin = f'{PRETRAIN_BIN}/train.bin'
if os.path.exists(train_bin):
    import numpy as np
    n = len(np.memmap(train_bin, dtype=np.uint16, mode='r'))
    print(f'✅ Données déjà tokenisées : {n:,} tokens (train)')
else:
    print('Tokenisation de Wikipedia FR...')
    prepare(
        input_path = WIKI_FILE,
        output_dir = PRETRAIN_BIN,
        val_ratio  = 0.01,
    )

import numpy as np
n_train = len(np.memmap(f'{PRETRAIN_BIN}/train.bin', dtype=np.uint16, mode='r'))
n_val   = len(np.memmap(f'{PRETRAIN_BIN}/val.bin',   dtype=np.uint16, mode='r'))
print(f'\n  Train : {n_train:,} tokens  |  Val : {n_val:,} tokens')
PRETRAIN_TRAIN = f'{PRETRAIN_BIN}/train.bin'
PRETRAIN_VAL   = f'{PRETRAIN_BIN}/val.bin'


In [ ]:
# @title 🔡 4.2 — Formatter et tokeniser PIAF (fine-tuning QA)
import json, os, numpy as np
import tiktoken

FINETUNE_BIN = f'{DATA_DIR}/finetune'
os.makedirs(FINETUNE_BIN, exist_ok=True)

ft_train_bin = f'{FINETUNE_BIN}/train.bin'
if os.path.exists(ft_train_bin):
    n = len(np.memmap(ft_train_bin, dtype=np.uint16, mode='r'))
    print(f'✅ PIAF déjà tokenisé : {n:,} tokens')
else:
    with open(PIAF_CACHE, encoding='utf-8') as f:
        piaf_data = json.load(f)

    enc = tiktoken.get_encoding('cl100k_base')

    # Format texte pour le LM causal :
    # Le modèle apprend à compléter "Réponse :" après avoir vu le contexte + question
    def format_qa(ex):
        return (
            f"### Contexte\n{ex['context']}\n\n"
            f"### Question\n{ex['question']}\n\n"
            f"### Réponse\n{ex['answer']}"
            f"<|endoftext|>"
        )

    print(f'Formatage et tokenisation de {len(piaf_data)} exemples PIAF...')
    all_tokens = []
    for ex in piaf_data:
        text   = format_qa(ex)
        tokens = enc.encode(text)
        all_tokens.extend(tokens)

    tokens_arr = np.array(all_tokens, dtype=np.uint16)
    n_val   = max(500, int(len(tokens_arr) * 0.05))
    n_train = len(tokens_arr) - n_val

    tokens_arr[:n_train].tofile(ft_train_bin)
    tokens_arr[n_train:].tofile(f'{FINETUNE_BIN}/val.bin')

    print(f'✅ PIAF tokenisé : train={n_train:,} | val={n_val:,} tokens')

# Aperçu du format
print(f'\nExemple de format QA :')
print('─'*50)
with open(PIAF_CACHE, encoding='utf-8') as f:
    ex = json.load(f)[0]
print(format_qa(ex)[:400])
print('─'*50)

FINETUNE_TRAIN = f'{FINETUNE_BIN}/train.bin'
FINETUNE_VAL   = f'{FINETUNE_BIN}/val.bin'


---
## 5️⃣ Pré-entraînement
> Entraîner le modèle depuis zéro sur Wikipedia FR.
> Les checkpoints sont sauvegardés automatiquement sur Drive.


In [ ]:
# @title ⚙️ 5.1 — Configuration du pré-entraînement
from config import TrainConfig, PRESETS
import torch

# ── À modifier selon ton GPU ───────────────────────────
PT_MODEL_SIZE = '50M'    # @param ['15M', '50M', '125M']
PT_MAX_ITERS  = 10000    # @param {type:'integer'}
PT_BATCH      = 8        # @param {type:'slider', min:1, max:32, step:1}
PT_ACCUM      = 8        # @param {type:'slider', min:1, max:32, step:1}
PT_SEQ_LEN    = 512      # @param [256, 512, 1024]
PT_LR         = 3e-4     # @param {type:'number'}
PT_COMPILE    = True     # @param {type:'boolean'}
# ─────────────────────────────────────────────────────

pt_cfg              = TrainConfig()
pt_cfg.model_size   = PT_MODEL_SIZE
pt_cfg.max_iters    = PT_MAX_ITERS
pt_cfg.batch_size   = PT_BATCH
pt_cfg.grad_accum   = PT_ACCUM
pt_cfg.seq_len      = PT_SEQ_LEN
pt_cfg.lr           = PT_LR
pt_cfg.min_lr       = PT_LR / 10
pt_cfg.warmup_iters = max(200, PT_MAX_ITERS // 20)
pt_cfg.compile      = PT_COMPILE
pt_cfg.out_dir      = CKPT_PRETRAIN
pt_cfg.data_path    = PRETRAIN_TRAIN
pt_cfg.val_path     = PRETRAIN_VAL
pt_cfg.log_every    = 50
pt_cfg.eval_every   = 500
pt_cfg.save_every   = 2000

mcfg = PRESETS[PT_MODEL_SIZE]
mcfg.max_seq_len = PT_SEQ_LEN

batch_eff = PT_BATCH * PT_ACCUM
tok_iter  = batch_eff * PT_SEQ_LEN
print(f'{'═'*55}')
print(f'  Pré-entraînement MiniLLM-{PT_MODEL_SIZE}')
print(f'{'═'*55}')
print(f'  Paramètres     : {mcfg.count_params()/1e6:.1f}M')
print(f'  Itérations     : {PT_MAX_ITERS:,}')
print(f'  Batch effectif : {PT_BATCH} × {PT_ACCUM} = {batch_eff}')
print(f'  Séquence       : {PT_SEQ_LEN} tokens')
print(f'  Tokens/iter    : {tok_iter:,}')
print(f'  Total tokens   : {PT_MAX_ITERS*tok_iter/1e6:.1f}M')
print(f'  Checkpoints    : {CKPT_PRETRAIN}')
vram = torch.cuda.get_device_properties(0).total_memory/1e9
req  = mcfg.count_params()*4*4/1e9
ok   = '✅' if vram > req else '⚠️ VRAM insuffisante'
print(f'  VRAM dispo     : {vram:.1f} GB  |  Requis ~{req:.1f} GB  {ok}')
print(f'{'═'*55}')


In [ ]:
# @title 🚀 5.2 — Lancer le pré-entraînement
# ⚠️  Exécuter 5.1 d'abord !
import os
from train import train

# Reprendre automatiquement si un checkpoint existe
best_ckpt = os.path.join(CKPT_PRETRAIN, 'best.pt')
if os.path.exists(best_ckpt):
    print(f'  → Reprise depuis : {best_ckpt}')
    pt_cfg.resume_from = best_ckpt

print('Démarrage du pré-entraînement...\n')
train(pt_cfg)
print(f'\n✅ Pré-entraînement terminé → {CKPT_PRETRAIN}/best.pt')


In [ ]:
# @title 📈 5.3 — Courbe de loss (pré-entraînement)
import torch, matplotlib.pyplot as plt, os, glob

ckpts = sorted(glob.glob(f'{CKPT_PRETRAIN}/*.pt'))
iters, losses = [], []
for path in ckpts:
    try:
        c = torch.load(path, map_location='cpu', weights_only=False)
        if c.get('val_loss'):
            iters.append(c.get('iter',0))
            losses.append(c['val_loss'])
    except: pass

if losses:
    plt.figure(figsize=(11,4))
    plt.plot(iters, losses, 'b-o', lw=2, ms=4, label='val loss')
    plt.xlabel('Itération'); plt.ylabel('Loss')
    plt.title(f'MiniLLM-{PT_MODEL_SIZE} — Pré-entraînement (Wikipedia FR)')
    plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
    plt.savefig(f'{DRIVE_BASE}/pretrain_loss.png', dpi=150)
    plt.show()
    print(f'  Meilleure val_loss : {min(losses):.4f}  (iter {iters[losses.index(min(losses))]})')
else:
    print('  Pas encore de checkpoints.')


---
## 6️⃣ Fine-tuning QA (PIAF)
> Spécialiser le modèle pré-entraîné sur les 3 835 paires Q/R françaises de PIAF.
> Le modèle apprend à lire un contexte, comprendre une question, et générer la réponse.


In [ ]:
# @title ⚙️ 6.1 — Configuration du fine-tuning
from config import TrainConfig, PRESETS

FT_MAX_ITERS  = 3000   # @param {type:'integer'}
FT_BATCH      = 4      # @param {type:'slider', min:1, max:16, step:1}
FT_ACCUM      = 4      # @param {type:'slider', min:1, max:16, step:1}
FT_SEQ_LEN    = 512    # @param [256, 512, 1024]
FT_LR         = 5e-5   # @param {type:'number'}
# LR plus faible que le pré-entraînement (fine-tuning)

ft_cfg              = TrainConfig()
ft_cfg.model_size   = PT_MODEL_SIZE   # même taille que le pré-entraînement
ft_cfg.max_iters    = FT_MAX_ITERS
ft_cfg.batch_size   = FT_BATCH
ft_cfg.grad_accum   = FT_ACCUM
ft_cfg.seq_len      = FT_SEQ_LEN
ft_cfg.lr           = FT_LR
ft_cfg.min_lr       = FT_LR / 10
ft_cfg.warmup_iters = 100
ft_cfg.compile      = True
ft_cfg.out_dir      = CKPT_FINETUNE
ft_cfg.data_path    = FINETUNE_TRAIN
ft_cfg.val_path     = FINETUNE_VAL
ft_cfg.log_every    = 50
ft_cfg.eval_every   = 300
ft_cfg.save_every   = 1000

# Repartir du meilleur checkpoint du pré-entraînement
import os
pt_best = os.path.join(CKPT_PRETRAIN, 'best.pt')
assert os.path.exists(pt_best), f'Checkpoint pré-entraînement introuvable : {pt_best}\nLance la section 5 d abord !'
ft_cfg.resume_from = pt_best

mcfg = PRESETS[PT_MODEL_SIZE]
mcfg.max_seq_len = FT_SEQ_LEN

print(f'{'═'*52}')
print(f'  Fine-tuning QA — MiniLLM-{PT_MODEL_SIZE}')
print(f'{'═'*52}')
print(f'  Depuis         : {pt_best}')
print(f'  Dataset        : PIAF (3 835 paires Q/R françaises)')
print(f'  Itérations     : {FT_MAX_ITERS:,}')
print(f'  Learning rate  : {FT_LR} (< LR pré-entraînement)')
print(f'  Checkpoints    : {CKPT_FINETUNE}')
print(f'{'═'*52}')


In [ ]:
# @title 🎯 6.2 — Lancer le fine-tuning
# ⚠️  Exécuter 6.1 d'abord !
from train import train
print('Démarrage du fine-tuning QA...\n')
train(ft_cfg)
print(f'\n✅ Fine-tuning terminé → {CKPT_FINETUNE}/best.pt')


In [ ]:
# @title 📈 6.3 — Courbe de loss (fine-tuning)
import torch, matplotlib.pyplot as plt, glob

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (ckpt_dir, label, color) in zip(axes, [
    (CKPT_PRETRAIN,  'Pré-entraînement (Wikipedia)', 'steelblue'),
    (CKPT_FINETUNE,  'Fine-tuning (PIAF QA)',         'darkorange'),
]):
    iters, losses = [], []
    for path in sorted(glob.glob(f'{ckpt_dir}/*.pt')):
        try:
            c = torch.load(path, map_location='cpu', weights_only=False)
            if c.get('val_loss'):
                iters.append(c.get('iter',0))
                losses.append(c['val_loss'])
        except: pass
    if losses:
        ax.plot(iters, losses, color=color, lw=2, marker='o', ms=3)
        ax.set_title(label); ax.set_xlabel('Iter'); ax.set_ylabel('Val Loss')
        ax.grid(True, alpha=0.3)
        ax.set_title(f'{label}\nMeilleure : {min(losses):.4f}')
    else:
        ax.text(0.5, 0.5, 'Pas de données', ha='center', transform=ax.transAxes)

plt.suptitle(f'MiniLLM-{PT_MODEL_SIZE} — Courbes d\'entraînement', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/all_losses.png', dpi=150)
plt.show()


---
## 7️⃣ Génération de texte
> Tester les deux modèles : pré-entraîné (texte libre) et fine-tuné (QA).


In [ ]:
# @title 📂 7.1 — Choisir le modèle à utiliser
import os
from generate import load_model

MODEL_TYPE = 'Fine-tuné QA (PIAF)'  # @param ['Pré-entraîné (Wikipedia)', 'Fine-tuné QA (PIAF)']

if MODEL_TYPE == 'Fine-tuné QA (PIAF)':
    ckpt_path = os.path.join(CKPT_FINETUNE, 'best.pt')
    mode_label = 'QA'
else:
    ckpt_path = os.path.join(CKPT_PRETRAIN, 'best.pt')
    mode_label = 'Texte libre'

assert os.path.exists(ckpt_path), f'Checkpoint introuvable : {ckpt_path}'
gen_model, gen_cfg, gen_device = load_model(ckpt_path)
print(f'✅ Modèle chargé : {MODEL_TYPE}  ({gen_model.n_params/1e6:.1f}M params)')


In [ ]:
# @title ✍️ 7.2 — Génération texte libre
import tiktoken
from generate import generate

PROMPT_LIBRE   = 'La France est un pays'  # @param {type:'string'}
MAX_TOKENS     = 150                       # @param {type:'slider', min:50, max:400, step:25}
TEMPERATURE    = 0.8                       # @param {type:'slider', min:0.1, max:1.5, step:0.1}
TOP_K          = 50                        # @param {type:'slider', min:0, max:100, step:5}

enc    = tiktoken.get_encoding('cl100k_base')
tokens = enc.encode(PROMPT_LIBRE)
out    = generate(gen_model, tokens, MAX_TOKENS, TEMPERATURE, TOP_K, device=gen_device)
text   = enc.decode(out)

print(f'{'─'*55}')
print(text)
print(f'{'─'*55}')
print(f'Tokens générés : {len(out)-len(tokens)}')


In [ ]:
# @title ❓ 7.3 — Mode QA interactif
# Fonctionne mieux avec le modèle fine-tuné (section 6)
import tiktoken
from generate import generate

CONTEXTE  = 'Paris est la capitale de la France. Elle est située au bord de la Seine.'  # @param {type:'string'}
QUESTION  = 'Quelle rivière traverse Paris ?'  # @param {type:'string'}
MAX_TOKENS_QA = 80  # @param {type:'slider', min:20, max:200, step:10}

enc    = tiktoken.get_encoding('cl100k_base')
prompt = (
    f'### Contexte\n{CONTEXTE}\n\n'
    f'### Question\n{QUESTION}\n\n'
    f'### Réponse\n'
)
tokens = enc.encode(prompt)
out    = generate(gen_model, tokens, MAX_TOKENS_QA, temperature=0.3, top_k=20, device=gen_device)
answer = enc.decode(out[len(tokens):])
# Couper au premier <|endoftext|> ou saut de ligne double
answer = answer.split('<|endoftext|>')[0].split('\n\n')[0].strip()

print(f'{'─'*55}')
print(f'Contexte  : {CONTEXTE}')
print(f'Question  : {QUESTION}')
print(f'{'─'*55}')
print(f'Réponse   : {answer}')
print(f'{'─'*55}')


---
## 8️⃣ Sauvegarde
> Exporter les modèles entraînés vers Drive ou les télécharger sur ton PC.


In [ ]:
# @title 💾 8.1 — Résumé des fichiers sur Drive
import os, glob, torch

print(f'{'═'*58}')
print(f'  Fichiers MiniLLM v2 sur Google Drive')
print(f'  Base : {DRIVE_BASE}')
print(f'{'═'*58}')
for pattern, label in [
    (f'{CKPT_PRETRAIN}/*.pt',    'Checkpoints pré-entraînement'),
    (f'{CKPT_FINETUNE}/*.pt',    'Checkpoints fine-tuning'),
    (f'{DATA_DIR}/**',            'Données'),
]:
    files = sorted(glob.glob(pattern, recursive=True))
    files = [f for f in files if os.path.isfile(f)]
    if files:
        total = sum(os.path.getsize(f) for f in files)/1e6
        print(f'\n  {label} ({total:.0f} MB) :')
        for f in files:
            size = os.path.getsize(f)/1e6
            print(f'    {os.path.basename(f):30s}  {size:.0f} MB')
print(f'{'═'*58}')


In [ ]:
# @title 📥 8.2 — Télécharger un modèle sur ton PC
from google.colab import files
import os

DOWNLOAD = 'Fine-tuné QA (best.pt)'  # @param ['Pré-entraîné (best.pt)', 'Fine-tuné QA (best.pt)']

if 'QA' in DOWNLOAD:
    path = os.path.join(CKPT_FINETUNE, 'best.pt')
else:
    path = os.path.join(CKPT_PRETRAIN, 'best.pt')

if os.path.exists(path):
    size_mb = os.path.getsize(path)/1e6
    print(f'Téléchargement : {path}  ({size_mb:.0f} MB)...')
    files.download(path)
else:
    print(f'❌ Fichier introuvable : {path}')


In [ ]:
# @title 📤 8.3 — Uploader un checkpoint existant
from google.colab import files
import shutil, os

DEST_TYPE = 'Fine-tuning'  # @param ['Pré-entraînement', 'Fine-tuning']
dest_dir  = CKPT_FINETUNE if DEST_TYPE == 'Fine-tuning' else CKPT_PRETRAIN

print('Sélectionne ton fichier .pt...')
uploaded = files.upload()
for fname in uploaded:
    dest = os.path.join(dest_dir, 'best.pt')
    shutil.move(fname, dest)
    print(f'✅ Uploadé → {dest}')
print('Tu peux maintenant aller à la section 7 pour générer du texte.')


---
## 🎁 Bonus — Conseils & prochaines étapes

### Améliorer les résultats

| Action | Impact |
|--------|--------|
| Plus de tokens Wikipedia (10%→100%) | ++++ |
| Augmenter les itérations | +++ |
| Modèle plus grand (125M) | +++ |
| Ajouter d'autres datasets FR | ++ |
| Entraîner un tokenizer custom (Fulfulde) | ++ |

### Ajouter tes propres données Fulfulde

```python
# Dans la cellule 4.1, remplacer WIKI_FILE par ton propre corpus :
# Combiner Wikipedia FR + tes données Fulfulde
import os
with open('corpus_combine.txt', 'w') as out:
    for src in ['wikipedia_fr.txt', 'corpus_fulfulde.txt']:
        if os.path.exists(src):
            with open(src) as f:
                out.write(f.read())
WIKI_FILE = 'corpus_combine.txt'
```

### Liens utiles

- 📦 Repo GitHub : https://github.com/bono-p/minillm_v2
- 📚 PIAF dataset : https://huggingface.co/datasets/etalab-ia/piaf
- 🌐 Wikipedia HF : https://huggingface.co/datasets/wikipedia
